# SAID — Spectral-Attention Integrated Detector
## Kaggle P100 Training Notebook

**Architecture:** YOLOv12-X + A2C2f-FSA (FFT fog disentanglement) + WIoU_v3_InnerMPDIoU loss  
**Target:** >93% mAP@0.5 on RTTS (baseline: 88.84%)

### Required Kaggle Datasets (add via sidebar → + Add Data)
| Dataset name on Kaggle | Content | Kaggle path |
|---|---|---|
| `rtts-ready` | RTTS_Ready/ (1 GB) | `/kaggle/input/rtts-ready/` |
| `voc-fog-yolo` | VOC_FOG_YOLO/ (578 MB) | `/kaggle/input/voc-fog-yolo/` |

> Enable **GPU P100** in Settings → Accelerator

In [ ]:
# ── Cell 1: Pull latest code from GitHub ─────────────────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path

REPO_URL  = 'https://github.com/habibour/foggy_object-detection_yolox.git'
REPO_DIR  = Path('/kaggle/working/repo')
WORK_DIR  = Path('/kaggle/working')

# Clone or pull latest
if REPO_DIR.exists():
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# Copy said/ module into working dir
shutil.copytree(str(REPO_DIR / 'said'), str(WORK_DIR / 'said'), dirs_exist_ok=True)
shutil.copy2(str(REPO_DIR / 'train.py'), str(WORK_DIR / 'train.py'))

# Add working dir to Python path
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

version = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'log', '--oneline', '-1']
).decode().strip()
print(f'\n✅ Code ready')
print(f'   Version : {version}')
print(f'   Files   : {list(WORK_DIR.glob("*.py")) + list((WORK_DIR/"said").glob("*.py"))}')

In [ ]:
# ── Cell 3: Verify datasets + write YAML configs ───────────────────────
from pathlib import Path

# ── Actual Kaggle dataset paths ──────────────────────────────────────
BASE = Path("/kaggle/input/datasets/mdhabibourrahman/object-detection-dataset")
RTTS_ROOT   = BASE / "RTTS_Ready"
VOCFOG_ROOT = BASE / "VOC_FOG_YOLO"
WORK_DIR    = Path("/kaggle/working")

assert RTTS_ROOT.exists(),   f"RTTS_Ready not found at {RTTS_ROOT}"
assert VOCFOG_ROOT.exists(), f"VOC_FOG_YOLO not found at {VOCFOG_ROOT}"

# Verify counts
for split in ["train", "val", "test"]:
    n = len(list((RTTS_ROOT / "images" / split).iterdir()))
    print(f"RTTS   images/{split}: {n} files")

for split in ["train", "val", "test"]:
    n = len(list((VOCFOG_ROOT / "images" / split).iterdir()))
    print(f"VOC-FOG images/{split}: {n} files")

# Write YAML configs
CLASSES = "  0: person\n  1: bicycle\n  2: car\n  3: bus\n  4: motorbike\n"

rtts_yaml = f"""path: {RTTS_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: bicycle
  2: car
  3: bus
  4: motorbike
"""

vocfog_yaml = f"""path: {VOCFOG_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: bicycle
  2: car
  3: bus
  4: motorbike
"""

(WORK_DIR / "rtts.yaml").write_text(rtts_yaml)
(WORK_DIR / "vocfog.yaml").write_text(vocfog_yaml)

print("✅ rtts.yaml   written")
print("✅ vocfog.yaml written")
print(f"
RTTS   root : {RTTS_ROOT}")
print(f"VOC-FOG root: {VOCFOG_ROOT}")


In [ ]:
# ── Cell 3: Verify datasets + write YAML configs ──────────────────────────────
import os
from pathlib import Path

RTTS_ROOT   = Path('/kaggle/input/rtts-ready')
VOCFOG_ROOT = Path('/kaggle/input/voc-fog-yolo')
WORK_DIR    = Path('/kaggle/working')

# Check datasets exist
for split in ['train', 'val', 'test']:
    n = len(list((RTTS_ROOT / 'images' / split).iterdir()))
    print(f'RTTS  images/{split}: {n} files')

for split in ['train', 'val', 'test']:
    n = len(list((VOCFOG_ROOT / 'images' / split).iterdir()))
    print(f'VOC-FOG images/{split}: {n} files')

# Write YAML configs pointing to Kaggle input paths
CLASSES = '  0: person\n  1: bicycle\n  2: car\n  3: bus\n  4: motorbike\n'

rtts_yaml = f'path: {RTTS_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n{CLASSES}'
vocfog_yaml = f'path: {VOCFOG_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n{CLASSES}'

(WORK_DIR / 'rtts.yaml').write_text(rtts_yaml)
(WORK_DIR / 'vocfog.yaml').write_text(vocfog_yaml)

print('\n✅ rtts.yaml   →', WORK_DIR / 'rtts.yaml')
print('✅ vocfog.yaml →', WORK_DIR / 'vocfog.yaml')

In [ ]:
# ── Cell 4: Sanity check custom modules ──────────────────────────────────────
import torch, sys
sys.path.insert(0, '/kaggle/working')

from said.a2c2f_fsa import A2C2f_FSA
from said.wiou_loss import WIoU_v3_InnerMPDIoU

m   = A2C2f_FSA(in_channels=64, out_channels=64, n_blocks=2, use_dsa=True)
x   = torch.randn(2, 64, 80, 80)
out = m(x)
assert out.shape == (2, 64, 80, 80)
print(f'✅ A2C2f_FSA  : {tuple(x.shape)} → {tuple(out.shape)}')

loss_fn = WIoU_v3_InnerMPDIoU(alpha=0.6, inner_scale=0.7)
pred    = torch.tensor([[10., 10., 50., 50.], [20., 20., 80., 80.]])
target  = torch.tensor([[12., 12., 52., 52.], [15., 15., 75., 75.]])
loss_fn.train()
loss = loss_fn(pred, target)
assert loss.item() >= 0
print(f'✅ WIoU_v3_InnerMPDIoU: loss = {loss.item():.4f}')
print('\nAll module checks passed — ready to train!')

In [ ]:
# ── Cell 5: Stage 1 — Pre-train on VOC-FOG (50 epochs) ───────────────────────
# Expected time on P100: ~2.5 hours
!cd /kaggle/working && python train.py \
    --stage vocfog \
    --s1-epochs 50 \
    --batch 32 \
    --device 0 \
    --kaggle

In [ ]:
# ── Cell 6: Stage 2 — Fine-tune on RTTS (100 epochs) ─────────────────────────
# Schedule: val/10ep | test/20ep | save-best/5ep
# Expected time on P100: ~2.5 hours
!cd /kaggle/working && python train.py \
    --stage rtts \
    --epochs 100 \
    --batch 32 \
    --device 0 \
    --val-freq 10 \
    --test-freq 20 \
    --save-freq 5 \
    --kaggle

In [ ]:
# ── Cell 7: Final evaluation ──────────────────────────────────────────────────
!cd /kaggle/working && python train.py \
    --stage validate \
    --batch 32 \
    --device 0 \
    --kaggle

In [ ]:
# ── Cell 8: List output files for download ────────────────────────────────────
from pathlib import Path
import json

work = Path('/kaggle/working')
print('=== Saved Weights ===')
for f in sorted(work.rglob('*.pt')):
    print(f'  {f.relative_to(work)}  ({f.stat().st_size/1e6:.1f} MB)')

print('\n=== Eval Log ===')
log = work / 'runs/said/stage2b_full/eval_log.csv'
if log.exists():
    print(log.read_text())

print('\n=== Summary ===')
summary = work / 'runs/said/stage2b_full/said_summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))